Exploring the dataset

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
import xgboost as xgb
import sklearn
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, balanced_accuracy_score
from sklearn.model_selection import PredefinedSplit, GridSearchCV
from sklearn.metrics import make_scorer, balanced_accuracy_score
import numpy as np
from sklearn.metrics import f1_score

In [ ]:
# simplified working path (notebook runs from `notebooks/`)
data_path = Path('..') / 'data' / 'BTCUSDT' / 'processed' / 'dataset_candle.parquet'
data_combined_path = Path('..') / 'data' / 'BTCUSDT' / 'processed' / 'dataset_combined.parquet'
data_orderbook_path = Path('..') / 'data' / 'BTCUSDT' / 'processed' / 'dataset_orderbook.parquet'
orderbook_features_path = Path('..') / 'data' / 'BTCUSDT' / 'processed' / 'orderbook_features.parquet'

print('reading', data_path.resolve())
print('reading', data_combined_path.resolve())
print('reading', data_orderbook_path.resolve())
print('reading', orderbook_features_path.resolve())

data = pd.read_parquet(data_path)
data_combined = pd.read_parquet(data_combined_path)
data_orderbook = pd.read_parquet(data_orderbook_path)
orderbook_features = pd.read_parquet(orderbook_features_path)

Exploration candle data

In [ ]:
print("Shape:", data.shape)
print("\nColumns:")
print(data.columns)

print("\nHead:")
print(data.head())
print("\nTail:")
print(data.tail())
print("\nMissing values per column:")
print(data.isna().sum())

print("\nNumeric summary:")
print(data.describe())

print("\nCategorical summary:")
print(data.describe(include="object"))

# Change Categorical data to numeric
obj_cols = ["quote_asset_volume", "taker_buy_base", "taker_buy_quote", "ignore"]
data[obj_cols] = data[obj_cols].apply(pd.to_numeric, errors="coerce")
numeric_cols = data.select_dtypes(include=[np.number]).columns.tolist()

# Remove close_time from numeric columns for plotting
numeric_cols.remove('close_time')

print("\nInfo:")
print(data.info())
plt.figure(figsize=(12, 8))
data[numeric_cols].boxplot()
plt.title("Boxplot of Numeric Features")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

Exploration orderbook data

In [ ]:
print("Shape:", data_orderbook.shape)
print("\nColumns:")
print(data_orderbook.columns)

print("\nHead:")
print(data_orderbook.head())

print("\nMissing values per column:")
print(data_orderbook.isna().sum())

print("\nNumeric summary:")
print(data_orderbook.describe())

numeric_cols = data_orderbook.select_dtypes(include=[np.number]).columns.tolist()

print("\nInfo:")
print(data_orderbook.info())
plt.figure(figsize=(12, 8))
data_orderbook[numeric_cols].boxplot()
plt.title("Boxplot of Numeric Features")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

Recalculate target value Quantile based

In [ ]:
data = data.sort_index().copy()

H = 1  # same horizon you used before

# 1) future return used for labeling
fr = data["close"].shift(-H) / data["close"] - 1

# 2) drop the last H rows (they have no future label)
data = data.iloc[:-H].copy()
fr = fr.iloc[:-H].copy()

# 3) define your split points on this trimmed dataset
gap = 5
n = len(data)
train_end = int(n * 0.7)
val_end   = int(n * 0.85)

# 4) compute quantile cutoffs on TRAIN ONLY
q_low, q_high = 1/3, 2/3  # or try 0.30/0.70 etc
q1 = fr.iloc[:train_end].quantile(q_low)
q2 = fr.iloc[:train_end].quantile(q_high)

# 5) assign quantile-based target
target_q = np.zeros(n, dtype=int)
target_q[fr.values <= q1] = -1
target_q[fr.values >= q2] = 1
data["target"] = target_q  # overwrite target

# (optional) inspect balance
print("Train balance:\n", pd.Series(target_q[:train_end]).value_counts(normalize=True).sort_index())
print("All balance:\n", data["target"].value_counts(normalize=True).sort_index())


XGBoost on candle data

In [ ]:
data = data.drop(columns=["ignore", "close_time"], errors="ignore")

X = data.drop(columns=['target'])
y_raw = data['target']                                                              
y= y_raw.map({-1: 0, 0: 1, 1: 2}).astype(int)  # Map -1 to 0, 0 to 1, and 1 to 2 for multiclass classification

gap = 5  # minutes
n = len(X)
train_end = int(n * 0.7)
val_end   = int(n * 0.85)

X_train, y_train = X.iloc[:train_end], y.iloc[:train_end]
X_val,   y_val   = X.iloc[train_end+gap:val_end], y.iloc[train_end+gap:val_end]
X_test,  y_test  = X.iloc[val_end+gap:], y.iloc[val_end+gap:]

class_counts = y_train.value_counts().sort_index()
class_weights = (class_counts.sum() / class_counts).to_dict()
w_train = y_train.map(class_weights).astype(float)


# Combine train + val into one "tuning" dataset
X_tune = np.vstack([X_train.values, X_val.values])
y_tune = np.concatenate([y_train.values, y_val.values])

# sample weights for val too (use train-derived weights or recompute on train+val)
class_counts = y_train.value_counts().sort_index()
class_weights = (class_counts.sum() / class_counts).to_dict()
w_val = y_val.map(class_weights).astype(float)
w_tune = np.concatenate([w_train.values, w_val.values])

# tell sklearn: -1 means "always train", 0 means "validation fold"
test_fold = np.r_[np.full(len(X_train), -1), np.zeros(len(X_val))]
ps = PredefinedSplit(test_fold)

base = xgb.XGBClassifier(
    objective="multi:softprob",
    num_class=3,
    tree_method="hist",
    random_state=42,
    n_jobs=-1,
    eval_metric="mlogloss",
)

scorer = make_scorer(balanced_accuracy_score)

param_grid = {
    "max_depth": [2, 3, 4],
    "min_child_weight": [5, 10, 20],
    "subsample": [0.7, 0.85, 1.0],
    "colsample_bytree": [0.7, 0.85, 1.0],
    "reg_lambda": [1.0, 5.0, 10.0],
    "gamma": [0.0, 0.1, 0.5],
    "learning_rate": [0.02, 0.05],
    "n_estimators": [400, 800, 1200],
}

grid = GridSearchCV(
    base, param_grid=param_grid, scoring=scorer, cv=ps,
    n_jobs=-1, verbose=2, refit=True
)

grid.fit(X_tune, y_tune, sample_weight=w_tune)

print("Best val score:", grid.best_score_)
print("Best params:", grid.best_params_)

best = grid.best_estimator_
pred = best.predict(X_test)

print("Balanced accuracy:", balanced_accuracy_score(y_test, pred))
# Best val score: 0.45427558606791596
# Best params: 
# Balanced accuracy: 0.4185724092488737{'colsample_bytree': 0.85, 'gamma': 0.0, 'learning_rate': 0.02, 'max_depth': 4, 'min_child_weight': 5, 'n_estimators': 400, 'reg_lambda': 1.0, 'subsample': 0.7}


# print("Balanced accuracy:", balanced_accuracy_score(y_test, pred))
# print(confusion_matrix(y_test, pred))
# print(classification_report(y_test, pred, digits=4, target_names=["down", "flat", "up"]))


In [ ]:
# #parameters = {'colsample_bytree': 0.85, 'gamma': 0.1, 'learning_rate': 0.02, 'max_depth': 3, 'min_child_weight': 10, 'n_estimators': 4000, 'reg_lambda': 5.0, 'subsample': 0.7}
# parameters = {'colsample_bytree': 0.85, 'gamma': 0.0, 'learning_rate': 0.03, 'max_depth': 4, 'min_child_weight': 5, 'n_estimators': 4000, 'reg_lambda': 1.0, 'subsample': 0.7}
# clf = xgb.XGBClassifier(
#     objective="multi:softprob",
#     num_class=3,
#     tree_method="hist",
#     random_state=42,
#     n_jobs=-1,
#     eval_metric="mlogloss",
#     early_stopping_rounds=100,
#     **parameters,
# )

# clf.fit(
#     X_train, y_train,
#     sample_weight=w_train,
#     eval_set=[(X_train, y_train), (X_val, y_val)],  
#     verbose=False,
# )

# predict = clf.predict(X_test)

# print("Balanced accuracy:", balanced_accuracy_score(y_test, predict))

# print("Best iteration:", clf.best_iteration)
# print("Best val mlogloss:", clf.best_score)



In [ ]:
# results = clf.evals_result()

# train_acc = 1.0 - np.array(results["validation_0"]["mlogloss"])
# val_acc  = 1.0 - np.array(results["validation_1"]["mlogloss"])


# best_it = clf.best_iteration  # chosen by early stopping (on validation_2)

# plt.figure()
# plt.plot(train_acc, label="train acc")
# plt.plot(val_acc, label="val acc")
# plt.axvline(best_it, linestyle="--", label=f"best iter={best_it}")
# plt.xlabel("Boosting round")
# plt.ylabel("Accuracy (1 - merror)")
# plt.legend()
# plt.show()

In [ ]:

decision_w = np.array([1.0, 0.7, 1.0])  # down, flat, up (your weights)
parameters = {'colsample_bytree': 0.85, 'gamma': 0.0, 'learning_rate': 0.03, 'max_depth': 4, 'min_child_weight': 5, 'reg_lambda': 1.0, 'subsample': 0.7}
# 1) Fit with early stopping on a smooth metric
es = xgb.XGBClassifier(
    objective="multi:softprob",
    num_class=3,
    tree_method="hist",
    n_estimators=5000,
    eval_metric="mlogloss",
    early_stopping_rounds=200,
    random_state=42,
    n_jobs=-1,
    # plus your tuned params:
    **parameters,  # <-- dict with max_depth, subsample, etc
)

es.fit(
    X_train, y_train,
    sample_weight=w_train,
    eval_set=[(X_val, y_val)],
    verbose=False,
)

# 2) Pick best iteration for *your* metric (weighted decision rule)
T = es.best_iteration + 1
scores = []
for t in range(1, T + 1):
    proba = es.predict_proba(X_val, iteration_range=(0, t))
    pred = (proba * decision_w[None, :]).argmax(axis=1)
    scores.append(balanced_accuracy_score(y_val, pred))

best_t = int(np.argmax(scores) + 1)
print("Best boosting round for weighted balanced acc:", best_t, "val:", max(scores))

# 3) Refit final model (no early stopping), using that number of trees
final = xgb.XGBClassifier(
    objective="multi:softprob",
    num_class=3,
    tree_method="hist",
    n_estimators=best_t,
    random_state=42,
    n_jobs=-1,
    **parameters,
)

# optional: train on train+val now that best_t is chosen
X_trval = np.concatenate([X_train.values, X_val.values], axis=0)
y_trval = np.concatenate([y_train.values, y_val.values], axis=0)

final.fit(X_trval, y_trval, verbose=False)

# test with the same decision rule
proba_test = final.predict_proba(X_test)
pred_test = (proba_test * decision_w[None, :]).argmax(axis=1)
print("Test weighted balanced acc:", balanced_accuracy_score(y_test, pred_test))

In [ ]:
results = es.evals_result()

val_acc  = np.array(results["validation_0"]["mlogloss"])

best_it = es.best_iteration  # chosen by early stopping (on validation_2)

plt.figure()

plt.plot(val_acc, label="val acc")
plt.axvline(best_it, linestyle="--", label=f"best iter={best_it}")
plt.xlabel("Boosting round")
plt.ylabel("Accuracy (1 - merror)")
plt.legend()
plt.show()

In [ ]:
proba_val = final.predict_proba(X_val)

def predict_weighted(proba, w):
    proba2 = proba * np.array(w)[None, :]
    proba2 = proba2 / proba2.sum(axis=1, keepdims=True)
    return proba2.argmax(axis=1)

best_w, best_score = None, -1
# Encourage down/up, discourage flat
for w_down in [1.2, 1.3, 1.4, 1.5, 1.6, 2.0, 2.5, 3.0]:
    for w_flat in [0.1, 0.2, 0.3, 0.4, 0.5, 0.6]:
        for w_up in [1, 1.1, 1.2, 1.3, 1.4, 1.5, 1.6, 1.7, 1.8, 1.9, 2, 3, 4]:
            pred = predict_weighted(proba_val, (w_down, w_flat, w_up))
            s = f1_score(y_val, pred, average="macro")
            if s > best_score:
                best_score, best_w = s, (w_down, w_flat, w_up)

print("best w (down,flat,up):", best_w, "val bal acc:", best_score)

# Apply to test
proba_test = best.predict_proba(X_test)
pred_test = predict_weighted(proba_test, best_w)
print("test bal acc:", balanced_accuracy_score(y_test, pred_test))

In [ ]:
print(confusion_matrix(y_test, pred_test))
print(classification_report(y_test, pred_test, digits=4, target_names=["down","flat","up"]))
print("training period", pd.Series(y_train.value_counts(normalize=True).sort_index()))
print("test period", pd.Series(y_test.value_counts(normalize=True).sort_index()))
print("validation period", pd.Series(y_val.value_counts(normalize=True).sort_index()))

XGBoost on orderbook data

In [ ]:

X = data_orderbook.drop(columns=['target'])
y_raw = data_orderbook['target']                                                              
y= y_raw.map({-1: 0, 0: 1, 1: 2}).astype(int)  # Map -1 to 0, 0 to 1, and 1 to 2 for multiclass classification

gap = 5  # minutes
n = len(X)
train_end = int(n * 0.7)
val_end   = int(n * 0.85)

X_train, y_train = X.iloc[:train_end], y.iloc[:train_end]
X_val,   y_val   = X.iloc[train_end+gap:val_end], y.iloc[train_end+gap:val_end]
X_test,  y_test  = X.iloc[val_end+gap:], y.iloc[val_end+gap:]

class_counts = y_train.value_counts().sort_index()
class_weights = (class_counts.sum() / class_counts).to_dict()
w_train = y_train.map(class_weights).astype(float)


# Combine train + val into one "tuning" dataset
X_tune = np.vstack([X_train.values, X_val.values])
y_tune = np.concatenate([y_train.values, y_val.values])

# sample weights for val too (use train-derived weights or recompute on train+val)
class_counts = y_train.value_counts().sort_index()
class_weights = (class_counts.sum() / class_counts).to_dict()
w_val = y_val.map(class_weights).astype(float)
w_tune = np.concatenate([w_train.values, w_val.values])

# tell sklearn: -1 means "always train", 0 means "validation fold"
test_fold = np.r_[np.full(len(X_train), -1), np.zeros(len(X_val))]
ps = PredefinedSplit(test_fold)

base = xgb.XGBClassifier(
    objective="multi:softprob",
    num_class=3,
    tree_method="hist",
    random_state=42,
    n_jobs=-1,
    eval_metric="mlogloss",
)

scorer = make_scorer(balanced_accuracy_score)

param_grid = {
    "max_depth": [2, 3, 4],
    "min_child_weight": [5, 10, 20],
    "subsample": [0.7, 0.85, 1.0],
    "colsample_bytree": [0.7, 0.85, 1.0],
    "reg_lambda": [1.0, 5.0, 10.0],
    "gamma": [0.0, 0.1, 0.5],
    "learning_rate": [0.02, 0.05],
    "n_estimators": [400, 800, 1200],
}

grid = GridSearchCV(
    base, param_grid=param_grid, scoring=scorer, cv=ps,
    n_jobs=-1, verbose=2, refit=True
)

grid.fit(X_tune, y_tune, sample_weight=w_tune)

print("Best val score:", grid.best_score_)
print("Best params:", grid.best_params_)

best = grid.best_estimator_
pred = best.predict(X_test)

print("Balanced accuracy:", balanced_accuracy_score(y_test, pred))